In [16]:
import time
import re
import pandas as pd
from scholarly import scholarly


In [17]:
def _clean_str(x):
    if x is None:
        return None
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x if x else None

def _to_int(x):
    try:
        return int(str(x).strip())
    except Exception:
        return None

def author_to_profile_df(author: dict) -> pd.DataFrame:
    prof = {
        "scholar_user_id": _clean_str(author.get("scholar_id")),
        "name": _clean_str(author.get("name")),
        "affiliation": _clean_str(author.get("affiliation")),
        "email_domain": _clean_str(author.get("email_domain")),
        "homepage": _clean_str(author.get("homepage")),
        "citedby": _to_int(author.get("citedby")),
        "citedby5y": _to_int(author.get("citedby5y")),
        "hindex": _to_int(author.get("hindex")),
        "hindex5y": _to_int(author.get("hindex5y")),
        "i10index": _to_int(author.get("i10index")),
        "i10index5y": _to_int(author.get("i10index5y")),
        "url_picture": _clean_str(author.get("url_picture")),
    }
    df = pd.DataFrame([prof])
    for c in ["citedby","citedby5y","hindex","hindex5y","i10index","i10index5y"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df

def author_to_pubs_df(author: dict) -> pd.DataFrame:
    uid = _clean_str(author.get("scholar_id"))
    pubs = author.get("publications", []) or []

    rows = []
    for p in pubs:
        bib = p.get("bib", {}) or {}
        rows.append({
            "scholar_user_id": uid,
            "author_name": _clean_str(author.get("name")),
            "author_affiliation": _clean_str(author.get("affiliation")),
            "title": _clean_str(bib.get("title")),
            "year": _to_int(bib.get("pub_year")),
            "citation": _clean_str(bib.get("citation")),
            "authors": _clean_str(bib.get("author")),
            "venue": _clean_str(bib.get("venue") or bib.get("journal") or bib.get("publisher")),
            "num_citations": _to_int(p.get("num_citations")),
            "citedby_url": _clean_str(p.get("citedby_url")),
            "author_pub_id": _clean_str(p.get("author_pub_id")),
            "filled": bool(p.get("filled", False)),
        })

    df = pd.DataFrame(rows)

    # --- types (robusto a NaN) ---
    if not df.empty:
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["num_citations"] = pd.to_numeric(df["num_citations"], errors="coerce").astype("Int64")

    # --- dedupe: author_pub_id if available, else title+year ---
    if not df.empty:
        df["pk"] = df["author_pub_id"]
        m = df["pk"].isna() | (df["pk"] == "")
        df.loc[m, "pk"] = (
            df.loc[m, "title"].fillna("").str.lower().str.strip()
            + "||"
            + df.loc[m, "year"].astype(str).replace("<NA>", "")
        )

        # si hay “mejor versión”, se queda con esa:
        # filled=True > filled=False, y luego más citas
        df["_cites"] = df["num_citations"].fillna(0).astype(int)

        df = df.sort_values(
            by=["scholar_user_id", "pk", "filled", "_cites"],
            ascending=[True, True, False, False]
        ).drop_duplicates(subset=["scholar_user_id", "pk"], keep="first")

        df = df.drop(columns=["pk", "_cites"]).reset_index(drop=True)

        # orden final “bonito”
        df = df.sort_values(
            by=["year", "num_citations", "title"],
            ascending=[False, False, True],
            na_position="last"
        ).reset_index(drop=True)

    return df

In [18]:

def fetch_all_authors(authors_gs: pd.DataFrame, sleep_s: float = 3.0):
    prof_list = []
    pubs_list = []
    errs = []

    for _, r in authors_gs.iterrows():
        uid = r.get("scholar_user_id")
        uid = None if pd.isna(uid) else str(uid).strip()

        if not uid:
            errs.append({
                "prof_id": r.get("prof_id"),
                "prof_name": r.get("prof_name"),
                "scholar_user_id": uid,
                "error": "missing scholar_user_id"
            })
            continue

        try:
            # --- fetch ---
            author = scholarly.search_author_id(uid)
            author = scholarly.fill(author)

            # --- process (FUNCIONES YA VALIDADAS) ---
            prof_df = author_to_profile_df(author)
            pubs_df = author_to_pubs_df(author)

            # --- attach institutional ids ---
            for c in ["prof_id", "prof_name"]:
                prof_df[c] = r.get(c)
                pubs_df[c] = r.get(c)

            prof_list.append(prof_df)
            pubs_list.append(pubs_df)

            time.sleep(sleep_s)

        except Exception as e:
            errs.append({
                "prof_id": r.get("prof_id"),
                "prof_name": r.get("prof_name"),
                "scholar_user_id": uid,
                "error": str(e)
            })

    prof_all = pd.concat(prof_list, ignore_index=True) if prof_list else pd.DataFrame()
    pubs_all = pd.concat(pubs_list, ignore_index=True) if pubs_list else pd.DataFrame()
    err_all = pd.DataFrame(errs)

    # --- consolidated view ---
    if not prof_all.empty and not pubs_all.empty:
        merged_all = pubs_all.merge(
            prof_all[[
                "scholar_user_id",
                "citedby", "hindex", "i10index",
                "affiliation", "homepage"
            ]].drop_duplicates(),
            on="scholar_user_id",
            how="left"
        )
    else:
        merged_all = pubs_all.copy()

    return prof_all, pubs_all, merged_all, err_all


In [ ]:
authors_gs = pd.DataFrame([
    {
        "prof_id": "TEC-001",
        "prof_name": "Molina-Perez, Edmundo",
        "scholar_user_id": "mD0NJJIAAAAJ"
    },
    {
        "prof_id": "TEC-002",
        "prof_name": "Syme, James",
        "scholar_user_id": "cOdbx8oAAAAJ"
    },
    {
        "prof_id": "TEC-003",
        "prof_name": "Bernal-Serrano, Daniel",
        "scholar_user_id": "FylAv6QAAAAJ"
    },
    {
        "prof_id": "TEC-004",
        "prof_name": "Aguilar-Gomez, Sandra",
        "scholar_user_id": "XjpOkmUAAAAJ"
    },
    {
        "prof_id": "TEC-005",
        "prof_name": "Campos-Rivera, Paola Abril",
        "scholar_user_id": "0ClZFNMAAAAJ"
    },
    {
        "prof_id": "TEC-006",
        "prof_name": "Garcia, Magdalena",
        "scholar_user_id": "BuYiG2kAAAAJ"
    },
    {
        "prof_id": "TEC-007",
        "prof_name": "Duran-Fernandez, Roberto",
        "scholar_user_id": "KSIpntkAAAAJ"
    },
    {
        "prof_id": "TEC-008",
        "prof_name": "Santos, Miguel Angel",
        "scholar_user_id": "PiWRreUAAAAJ"
    },
    # {
    #     "prof_id": "TEC-009",
    #     "prof_name": "Popper, Steven W.",
    #     "scholar_user_id": ""
    # },
    {
        "prof_id": "TEC-010",
        "prof_name": "Peraza-Mues, Gonzalo G.",
        "scholar_user_id": "nguMmzcAAAAJ"
    },
    {
        "prof_id": "TEC-011",
        "prof_name": "Sobrino, Fernanda",
        "scholar_user_id": "IL2Xlf0AAAAJ"
    },
    {
        "prof_id": "TEC-012",
        "prof_name": "Benavides-Rincon, Guillermina",
        "scholar_user_id": "cyVK7ngAAAAJ"
    },
    {
        "prof_id": "TEC-013",
        "prof_name": "Morales-Arilla, Jose",
        "scholar_user_id": "YwTag-gAAAAJ"
    },
    {
        "prof_id": "TEC-014",
        "prof_name": "Contreras-Loya, David",
        "scholar_user_id": "axuzNJ0AAAAJ"
    },
    {
        "prof_id": "TEC-015",
        "prof_name": "Gomez-Zaldivar, Fernando",
        "scholar_user_id": "qAgvXfQAAAAJ"
    },
    {
        "prof_id": "TEC-016",
        "prof_name": "Stein, Ernesto H.",
        "scholar_user_id": "EobFjNEAAAAJ"
    },
    {
        "prof_id": "TEC-017",
        "prof_name": "Diaz-Dominguez, Alejandro",
        "scholar_user_id": "s3yl5UQAAAAJ"
    },
    {
        "prof_id": "TEC-018",
        "prof_name": "Ponce-Lopez, Roberto",
        "scholar_user_id": "MJ3YExwAAAAJ"
    },
    {
        "prof_id": "TEC-019",
        "prof_name": "Silverio-Murillo, Adan",
        "scholar_user_id": "LdrxBQEAAAAJ"
    },
    {
        "prof_id": "TEC-020",
        "prof_name": "Villarreal, Héctor J.",
        "scholar_user_id": "aPTg0cQAAAAJ"
    },
    {
        "prof_id": "TEC-021",
        "prof_name": "De Unánue, Adolfo",
        "scholar_user_id": "Pv5NR2YAAAAJ"
    },
    {
        "prof_id": "TEC-021",
        "prof_name": "De Unánue, Adolfo",
        "scholar_user_id": "Pv5NR2YAAAAJ"
    },
])

In [21]:
prof_all, pubs_all, merged_all, err_all = fetch_all_authors(
    authors_gs,
    sleep_s=3.0 
)

prof_all.to_csv("google_scholar/scholar_profiles_all.csv", index=False)
pubs_all.to_csv("google_scholar/scholar_publications_all.csv", index=False)
merged_all.to_csv("google_scholar/scholar_publications_consolidated.csv", index=False)
err_all.to_csv("google_scholar/scholar_errors.csv", index=False)